# Decoder from Scratch

### Check version

In [13]:
from importlib.metadata import version
print("torch version: ", version("torch"))

torch version:  2.7.0+cu128


### Previous implementations

Multi-Head Attention, Feed Forward Network, LayerNorm, Encoder

In [14]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

import math

In [15]:
# form a dataclass ModelArgs
@dataclass
class ModelArgs:
    max_T: int
    D: int
    H: int 
    hidden_D: int
    dropout: float
    n_layers: int

In [16]:
# MultiHeadAttention
class MultiHeadAttention(nn.Module):
    
    def __init__(self, args: ModelArgs, is_causal = False):
        # inherit from nn.Module
        super().__init__()

        # set parameters
        self.H = args.H
        self.D = args.D
        assert self.D % self.H == 0
        self.D_h = self.D // self.H # D_h is each head embedding dimension
        self.is_causal = is_causal
        
        # linear transformtion layers
        # shape for Wq, Wk, Wv, Wo = [D, D]
        self.Wq = nn.Linear(self.D, self.D, bias = False)
        self.Wk = nn.Linear(self.D, self.D, bias = False)
        self.Wv = nn.Linear(self.D, self.D, bias = False)
        self.Wo = nn.Linear(self.D, self.D, bias = False)

        # dropout layer
        self.res_dropout = nn.Dropout(args.dropout)

        # mask
        if is_causal:
            mask = torch.full(
                (1, 1, args.max_T, args.max_T), 
                float("-inf"))
            mask = torch.triu(mask, diagonal=1)
            self.register_buffer("mask", mask) # register_buffer(name, tensor) tells the program this is not a model parameter
    
    def forward(self, q, k, v):
        # q, k, v shape = [B, T, D]
        B, T = q.shape[0], q.shape[1] 
        D = self.D
        H, D_h = self.H, self.D_h

        # step 1: linear transformtion with W
        # [B, T, D] * [D, D] -> [B, T, D]
        # shape for Q, K, V = [B, T, D]
        Q = self.Wq(q)
        K = self.Wk(k)
        V = self.Wv(v)

        # step 2: split into multiple heads
        # [B, T, D] = [B, T, H, D_h] where D = H * D_h
        # shape for Q, K, V = [B, T, H, D_h]
        Q = Q.view(B, T, H, D_h)
        K = K.view(B, T, H, D_h)
        V = V.view(B, T, H, D_h)

        # step 3: switch positions for T and H
        # [B, T, H, D_h] -> [B, H, T, D_h]
        # shape for Q, K, V = [B, H, T, D_h]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # step 4: compute attention score S
        # S = (Q K^T) / sqrt(D_h)
        # shape for S = [B, H, T, D_h] * [B, H, D_h, T] -> [B, H, T, T]
        S = torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(D_h)

        # step 5: mask
        # shape for S = [B, H, T, T]
        if self.is_causal:
            S = S + self.mask[:, :, :T, :T]

        # step 6: compute attention probabiloity S_prob by softmax
        # normalize the last dimension T
        # shape for S_prob = [B, H, T, T]
        S_prob = F.softmax(S, dim=-1).type_as(Q)

        # step 7: compute weighted V
        # S_prob * V -> [B, H, T, T] * [B, H, T, D_h] -> [B, H, T, D_h]
        # shape for output: [B, H, T, D_h]
        output = torch.matmul(S_prob, V)

        # step 8: concatenate multiple heads
        # [B, H, T, D_h] -> [B, T, H, D_h] -> [B, T, D]
        # shape for output: [B, T, D]
        output = output.transpose(1, 2).contiguous().view(B, T, D)

        # step 9: final linear transformation layer + residual dropout
        # [B, T, D] * [D, D] -> [B, T, D]
        # shape for output: [B, T, D]
        output = self.Wo(output)
        output = self.res_dropout(output)

        return output

In [17]:
# FFN
class FFN(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.w1 = nn.Linear(args.D, args.hidden_D, bias=False)
        self.w2 = nn.Linear(args.hidden_D, args.D, bias=False)
        self.dropout = nn.Dropout(args.dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.relu(self.w1(x))))

In [18]:
# LayerNorm
class LayerNorm(nn.Module):
    """
    formula:
    output = ((x - mean(x)) / (std(x) + eps)) * gamma + beta
    gamma and beta are learned parameters
    """
    def __init__(self, args: ModelArgs, eps=1e-6):
        super().__init__()
        
        # define gamma and beta
        self.gamma = nn.Parameter(torch.ones(args.D))
        self.beta = nn.Parameter(torch.zeros(args.D))

        self.eps = eps

    def forward(self, x):
        # x shape = [B, T, D], normalize within the last dimension D
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)

        return (x - mean) / (std + self.eps) * self.gamma + self.beta

In [23]:
# encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()

        self.norm1 = LayerNorm(args)
        self.attention = MultiHeadAttention(args, is_causal=False)

        self.norm2 = LayerNorm(args)
        self.ffn = FFN(args)

    def forward(self, x):
        x = self.norm1(x)
        x = x + self.attention(x, x, x)
        x = self.norm2(x)
        output = x + self.ffn(x)
        return output

In [24]:
# encoder
class Encoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()

        # multiple layers
        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layers)])

        self.norm = LayerNorm(args)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        
        output = self.norm(x)
        return output

### Decoder implementation

In [29]:
class DecoderLayer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()

        self.norm1 = LayerNorm(args)
        self.mask_attention = MultiHeadAttention(args, is_causal = True)

        self.norm2 = LayerNorm(args)
        self.attention = MultiHeadAttention(args, is_causal=False)

        self.norm3 = LayerNorm(args)
        self.ffn = FFN(args)

    def forward(self, x, encoder_x):
        x = self.norm1(x)
        x = x + self.mask_attention(x, x, x)

        x = self.norm2(x)
        x = x + self.attention(x, encoder_x, encoder_x)

        x = self.norm3(x)
        output = x + self.ffn(x)

        return output

In [30]:
class Decoder(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()

        self.layers = nn.ModuleList([DecoderLayer(args) for _ in range(args.n_layers)])

        self.norm = LayerNorm(args)

    def forward(self, x, encoder_x):
        for layer in self.layers:
            x = layer(x, encoder_x)

        return self.norm(x)

        


### Test

In [27]:
# hyperparameters
B = 10
H = 8
D = 768
hidden_D = D * 4
dropout = 0.1
max_T = 512
n_layers = 6

# instantiate args
args = ModelArgs(max_T = max_T, D = D, H = H, hidden_D = hidden_D, dropout = dropout, n_layers = n_layers)

# input
x = torch.randn(B, max_T, D)

# test encoder
encoder = Encoder(args)
encoder_output = encoder(x)

print("encoder output shape: ", encoder_output.shape)

encoder output shape:  torch.Size([10, 512, 768])


In [32]:
# test decoder
decoder = Decoder(args)
decoder_output = decoder(x, encoder_output)

print("decoder output shape: ", decoder_output.shape)

decoder output shape:  torch.Size([10, 512, 768])


In [33]:
print("decoder output: ", decoder_output)

decoder output:  tensor([[[-0.4089, -1.6218,  1.0308,  ...,  0.7608, -0.4148,  0.3252],
         [ 0.4133, -1.1134,  0.2922,  ...,  0.3980,  0.0718,  0.4994],
         [-0.7098, -1.8855,  0.0835,  ..., -0.5592, -0.6506, -0.5695],
         ...,
         [ 0.3684, -0.1506,  0.6983,  ..., -0.0182, -0.7990, -0.6830],
         [ 0.8690, -1.1694,  0.1799,  ...,  1.4997, -0.3323, -2.6043],
         [ 1.3912, -1.1396,  0.5117,  ...,  0.4516, -0.1450, -0.6505]],

        [[-0.5674, -0.6465,  1.0250,  ...,  0.2968, -1.2387,  0.4883],
         [-0.3045,  0.0226, -0.7526,  ...,  0.4695, -0.7436, -0.4631],
         [ 1.5633, -1.6560,  0.5563,  ..., -0.8021,  0.0389,  0.8547],
         ...,
         [ 1.5844, -0.7994,  1.1019,  ..., -0.8168, -0.9018, -1.8107],
         [-0.4604, -1.5783, -0.1754,  ...,  1.8148,  0.5567, -1.0721],
         [-0.3834,  0.3198,  0.0981,  ..., -0.3593, -0.0905,  0.0089]],

        [[ 1.5834,  0.4772,  1.2968,  ...,  0.6846, -0.7060, -0.4293],
         [-0.6947,  0.4345, 